<h2>Data Manipulation with Pandas</h2>

Pandas is a package built on top of NumPy, and provides an efficient implementation of a `DataFrame`. `DataFrames` are essentially multidimensional arrays with attached row and column labels, and often with heterogenous types and/or missing data. As well as offering a convenient storage interface for labeled data, Pandas implements a number of powerful data operations familiar to users of both database frameworks and spreadsheet programs.

NumPy's `ndarray` data structure provides essential features for the type of clean, well-organized data typically seen in numerical computing tasks. While it serves this purpose very well, its limitations become clear when we need more flexibility (attaching labels to data, working with missing data, etc.) and when attempting operations that do not map well to element-wise broadcasting (groupings, pivots, etc.), each of which is an important piece of analyzing the less structured data available in many forms in the world around us.

We'll explore the mechanics of using `Series`, `DataFrame`, and related structures effectively. These examples are from real datasets where appropriate.

In [1]:
import pandas

pandas.__version__

'1.2.4'

In [3]:
import pandas as pd

# Recall that you can easily access documentation with a question mark, or pd.<TAB>
pd?

Type:        module
String form: <module 'pandas' from '/Users/mike/opt/anaconda3/lib/python3.8/site-packages/pandas/__init__.py'>
File:        ~/opt/anaconda3/lib/python3.8/site-packages/pandas/__init__.py
Docstring:  
pandas - a powerful data analysis and manipulation library for Python

**pandas** is a Python package providing fast, flexible, and expressive data
structures designed to make working with "relational" or "labeled" data both
easy and intuitive. It aims to be the fundamental high-level building block for
doing practical, **real world** data analysis in Python. Additionally, it has
the broader goal of becoming **the most powerful and flexible open source data
analysis / manipulation tool available in any language**. It is already well on
its way toward this goal.

Main Features
-------------
Here are just a few of the things that pandas does well:

  - Easy handling of missing data in floating point as well as non-floating
    point data.
  - Size mutability: columns can 

In [5]:
import numpy as np

data = pd.Series([0.25, 0.5, 0.75, 1.0])
data

0    0.25
1    0.50
2    0.75
3    1.00
dtype: float64

The `Series` wraps both a sequence of values and a sequence of indices, which we can access with the `values` and `index` attributes. The `values` are simply a familiar NumPy array.

In [6]:
data.values

array([0.25, 0.5 , 0.75, 1.  ])

In [7]:
data.index

RangeIndex(start=0, stop=4, step=1)

In [8]:
type(data.values)

numpy.ndarray

Like with a NumPy array, data can be accessed by the associated index via the familiar Python square-bracket notation:

In [9]:
data[1]

0.5

In [10]:
data[1:3]

1    0.50
2    0.75
dtype: float64

The Pandas `Series` is much more general and flexible than the one-dimensional NumPy array that it emulates.

From what we've seen so far, it may look like the `Series` object is basically interchangeable with a one-dimensional NumPy array. The essential difference is the presence of the index: while the NumPy array has an <i>implicitly defined</i> integer index used to access the values, the Pandas `Series` has an <i>explicitly defined</i> index associated with the values.

The explicit index definition gives the `Series` object additional capabilities. For example, the index need not be an integer, but can consist of values of any desired type. Example of using strings as an index:

In [11]:
data = pd.Series([0.25, 0.5, 0.75, 1.0], index=['a', 'b', 'c', 'd'])
data

a    0.25
b    0.50
c    0.75
d    1.00
dtype: float64

In [12]:
data['b']

0.5

We can even use noncontiguous or nonsequential indices:

In [17]:
data = pd.Series([0.25, 0.5, 0.75, 1.0], index=[2, 5, 3, 7])
data

2    0.25
5    0.50
3    0.75
7    1.00
dtype: float64

In [18]:
data[5]

0.5

<b>Series as a specialized dictionary</b>

You can think of a Pandas `Series` a bit like a specialization of a Python dictionary. A dictionary is a structure that maps arbitrary keys to a set of arbitrary values, and a `Series` is a structure that maps typed keys to a set of typed values. <b>This typing is very important: just as the type-specific compiled code behind a NumPy array makes it more efficient than a Python list for certain operations, the type information of a Pandas `Series` makes it much more efficient than Python dictionaries for certain operations.</b>

We can make the Series-as-dictionary analogy even more clear by constructing a `Series` object directly from a Python dictionary:

In [21]:
population_dict = {'California': 38332521,
                   'Texas': 26448193,
                   'New York': 19651127,
                   'Florida': 19552860,
                   'Illinois': 12882135
                  }

population = pd.Series(population_dict)
population

California    38332521
Texas         26448193
New York      19651127
Florida       19552860
Illinois      12882135
dtype: int64

By default, a `Series` will be created where the index is drawn from the sorted keys. From here, a typical dictionary-style item access can be performed:

In [22]:
population['California']

38332521

Unlike a dictionary, however, the `Series` also supports array-style operations such as slicing:

In [23]:
population['California':'Illinois']

California    38332521
Texas         26448193
New York      19651127
Florida       19552860
Illinois      12882135
dtype: int64

<h3>Constructing Series Objects</h3>

Pandas `Series` creations are all some version of the following:
```python
>>> pd.Series(data, index=index)
```
where index is an optional argument, and data can be one of many entities.

For example, data can be a list or NumPy array, in which case index defaults to an integer sequence:

In [24]:
pd.Series([2, 4, 6])

0    2
1    4
2    6
dtype: int64

* data can be a scalar, which is repeated to fill the specified index:

In [25]:
pd.Series(5, index=[100, 200, 300])

100    5
200    5
300    5
dtype: int64

* data can be a dictionary, in which index defaults to the sorted dictionary keys:

In [26]:
pd.Series({2: 'a', 1: 'b', 3: 'c'})

2    a
1    b
3    c
dtype: object

* Note how in this case, the `Series` is only populated with the explicitly identified keys:

In [28]:
pd.Series({2: 'a', 1: 'b', 3: 'c'}, index=[3, 2])

3    c
2    a
dtype: object

<h3>The Pandas DataFrame Object</h3>

The next fundamental structure in Pandas is the `DataFrame`. Like the `Series` object discussed in the previous section, the `DataFrame` can be thought of either as a generalization of a NumPy array, or as a specialization of a Python dictionary. We'll now take a look at each of these perspectives.

<h4>DataFrame as a generalized NumPy array</h4>

If a `Series` is an analog of a one-dimensional array with flexible indices, a `DataFrame` is an analog of a two-dimensional array with both flexible row indices and flexible column names. Just as you might think of a two-dimensional array as an ordered sequence of aligned one-dimensional columns, <b>you can think of a `DataFrame` as a sequence of aligned `Series` objects. Here, by "aligned", we mean that they share the same index.</b>

To demonstrate this, let's first construct a new `Series` listing the area of each of the five states discussed in the previous section:

In [33]:
area_dict = {'California': 423967, 'Texas': 695662, 'New York': 141297,
             'Florida': 170312, 'Illinois': 149995}
area = pd.Series(area_dict)
area

California    423967
Texas         695662
New York      141297
Florida       170312
Illinois      149995
dtype: int64

* Now that we have this along with the `population Series` from before, we can use a dictionary to construct a single two-dimensional object containing this information:

In [34]:
states = pd.DataFrame({'population': population, 'area': area})
states

,population,area
California,38332521,423967
Texas,26448193,695662
New York,19651127,141297
Florida,19552860,170312
Illinois,12882135,149995


* Like the `Series` object, the `DataFrame` has an index attribute that gives access to the index labels. It also has a columns attribute:

In [35]:
states.index

Index(['California', 'Texas', 'New York', 'Florida', 'Illinois'], dtype='object')

In [36]:
states.columns

Index(['population', 'area'], dtype='object')

Thus the `DataFrame` can be thought of as a generalization of a two-dimensional NumPy array, where both the rows and columns have a generalized index for accessing the data.

<h3>DataFrame as a specialized dictionary</h3>

We can also think of a `DataFrame` as a specialization of a dictionary. Where a dictionary maps a key to a value, a `DataFrame` maps a column name to a `Series` of column data. For example, asking for the 'area' attribute returns the `Series` object containing the areas we saw earlier:

In [37]:
states['area']

California    423967
Texas         695662
New York      141297
Florida       170312
Illinois      149995
Name: area, dtype: int64

In [38]:
type(states['area'])

pandas.core.series.Series

Note the potential confusion here: in a two-dimensional NumPy array, `data[0]` will return the first <i>row</i>. For a `DataFrame`, `data['col0']` will return the first <i>column</i>. Because of this, it is probably better to think about `DataFrames` as generalized dictionaries rather than generalized arrays, though both ways of looking at the situation can be useful. We'll explore more flexible means of indexing `DataFrames` in a bit.

<h4>Constructing DataFrame objects</h4>

A Pandas `DataFrame` object can be constructed in a variety of ways. Here are several examples:

* <b>From a single `Series` object.</b> A `DataFrame` is a collection of `Series` objects, and a single-column `DataFrame` can be constructed from a single `Series`:

In [39]:
pd.DataFrame(population, columns=['population'])

,population
California,38332521
Texas,26448193
New York,19651127
Florida,19552860
Illinois,12882135


* <b>From a list of dicts</b>: Any list of dictionaries can be made into a `DataFrame`. We'll use a simple list comprehension to create some data:

In [40]:
data = [{'a': i, 'b': 2 * i} for i in range(3)]
data

[{'a': 0, 'b': 0}, {'a': 1, 'b': 2}, {'a': 2, 'b': 4}]

In [41]:
pd.DataFrame(data)

,a,b
0,0,0
1,1,2
2,2,4


* Even if some keys in the dictionary are missing, Pandas will fill them with `NaN` values:

In [42]:
pd.DataFrame([{'a': 1, 'b': 2}, {'b': 3, 'c': 4}])

,a,b,c
0,1.0,2,NaN
1,NaN,3,4.0


* <b>From a dictionary of Series objects</b>: A `DataFrame` can be constructed from a dictionary of `Series` objects as well:

In [43]:
pd.DataFrame({'population': population,
              'area': area})

,population,area
California,38332521,423967
Texas,26448193,695662
New York,19651127,141297
Florida,19552860,170312
Illinois,12882135,149995


* <b>From a two-dimensional NumPy array</b>: Given a two-dimensional array of data, we can create a `DataFrame` with any specified column and index names. If omitted, an integer index will be used for each:

In [44]:
pd.DataFrame(np.random.rand(3, 2),
             columns=['foo', 'bar'],
             index=['a', 'b', 'c']
            )

,foo,bar
a,0.165192,0.734091
b,0.633021,0.490757
c,0.316198,0.609857


* <b>From a NumPy structured array</b>: A Pandas `DataFrame` operates much like a structured array, and can be created directly from one: